<a href="https://colab.research.google.com/github/nishita-ds/pyTorch-learning/blob/main/14-lstm-using-pytorch/Next_word_predictor_using_pytorch_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
!pip install nltk

In [2]:
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
import nltk
from torch.utils.data import Dataset,DataLoader
from nltk.tokenize import word_tokenize

In [3]:
document = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""

In [4]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
# tokenize
tokens=word_tokenize(document.lower())

In [6]:
# build vocab
vocab={'<unk>':0}
Counter(tokens)
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token]=len(vocab)

vocab

{'<unk>': 0,
 'about': 1,
 'the': 2,
 'program': 3,
 'what': 4,
 'is': 5,
 'course': 6,
 'fee': 7,
 'for': 8,
 'data': 9,
 'science': 10,
 'mentorship': 11,
 '(': 12,
 'dsmp': 13,
 '2023': 14,
 ')': 15,
 'follows': 16,
 'a': 17,
 'monthly': 18,
 'subscription': 19,
 'model': 20,
 'where': 21,
 'you': 22,
 'have': 23,
 'to': 24,
 'make': 25,
 'payments': 26,
 'of': 27,
 'rs': 28,
 '799/month': 29,
 '.': 30,
 'total': 31,
 'duration': 32,
 '?': 33,
 '7': 34,
 'months': 35,
 'so': 36,
 'becomes': 37,
 '799': 38,
 '*': 39,
 '=': 40,
 '5600': 41,
 'approx': 42,
 'syllabus': 43,
 'we': 44,
 'will': 45,
 'be': 46,
 'covering': 47,
 'following': 48,
 'modules': 49,
 ':': 50,
 'python': 51,
 'fundamentals': 52,
 'libraries': 53,
 'analysis': 54,
 'sql': 55,
 'maths': 56,
 'machine': 57,
 'learning': 58,
 'ml': 59,
 'algorithms': 60,
 'practical': 61,
 'mlops': 62,
 'case': 63,
 'studies': 64,
 'can': 65,
 'check': 66,
 'detailed': 67,
 'here': 68,
 '-': 69,
 'https': 70,
 '//learnwith.campusx.i

In [7]:
len(vocab)

289

In [8]:
# extract sentences from data
sentences=[]
for sentence in document.split('\n'):
  sentences

In [9]:
sentences=document.split('\n')

In [10]:
sentences

['About the Program',
 'What is the course fee for  Data Science Mentorship Program (DSMP 2023)',
 'The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.',
 'What is the total duration of the course?',
 'The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)',
 'What is the syllabus of the mentorship program?',
 'We will be covering the following modules:',
 'Python Fundamentals',
 'Python libraries for Data Science',
 'Data Analysis',
 'SQL for Data Science',
 'Maths for Machine Learning',
 'ML Algorithms',
 'Practical ML',
 'MLOPs',
 'Case studies',
 'You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390',
 'Will Deep Learning and NLP be a part of this program?',
 'No, NLP and Deep Learning both are not a part of this program’s curriculum.',
 'What if I miss a live session? Will I get a recording 

In [11]:
for sentence in sentences:
  print(word_tokenize(sentence.lower()))

['about', 'the', 'program']
['what', 'is', 'the', 'course', 'fee', 'for', 'data', 'science', 'mentorship', 'program', '(', 'dsmp', '2023', ')']
['the', 'course', 'follows', 'a', 'monthly', 'subscription', 'model', 'where', 'you', 'have', 'to', 'make', 'monthly', 'payments', 'of', 'rs', '799/month', '.']
['what', 'is', 'the', 'total', 'duration', 'of', 'the', 'course', '?']
['the', 'total', 'duration', 'of', 'the', 'course', 'is', '7', 'months', '.', 'so', 'the', 'total', 'course', 'fee', 'becomes', '799', '*', '7', '=', 'rs', '5600', '(', 'approx', '.', ')']
['what', 'is', 'the', 'syllabus', 'of', 'the', 'mentorship', 'program', '?']
['we', 'will', 'be', 'covering', 'the', 'following', 'modules', ':']
['python', 'fundamentals']
['python', 'libraries', 'for', 'data', 'science']
['data', 'analysis']
['sql', 'for', 'data', 'science']
['maths', 'for', 'machine', 'learning']
['ml', 'algorithms']
['practical', 'ml']
['mlops']
['case', 'studies']
['you', 'can', 'check', 'the', 'detailed', 'sy

In [12]:
def text_to_indices(sentence,vocab):
  numerical_sentence=[]
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])
  return numerical_sentence

In [13]:
numerical_sentences=[]
for sentence in sentences:
  numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()),vocab))


In [14]:
len(numerical_sentences)

78

In [15]:
training_sequence=[]
for sentence in numerical_sentences:
  for i in range(1,len(sentence)):
    training_sequence.append(sentence[:i+1])




In [16]:
len(training_sequence)

942

In [17]:
len_list=[]
for sequence in training_sequence:
  len_list.append(len(sequence))
max(len_list)


62

In [18]:
padded_training_sequence=[]
for sequence in training_sequence:
  padded_training_sequence.append([0]*(max(len_list) - len(sequence))+sequence)

In [19]:
len(padded_training_sequence[10])

62

In [20]:
padded_training_sequence=torch.tensor(padded_training_sequence,dtype=torch.long)

In [21]:
padded_training_sequence.shape

torch.Size([942, 62])

In [22]:
x=padded_training_sequence[:,:-1]
y=padded_training_sequence[:,-1]

In [23]:
class CustomDataset(Dataset):
  def __init__(self,x,y):
    self.x=x
    self.y=y
  def __len__(self):
    return self.x.shape[0]
  def __getitem__(self,idx):
    return self.x[idx],self.y[idx]

In [24]:
dataset=CustomDataset(x,y)

In [25]:
len(dataset)

942

In [26]:
dataloader=DataLoader(dataset,batch_size=32,shuffle=True)

In [27]:
for input,output in dataloader:
  print(input,output)

tensor([[  0,   0,   0,  ...,   2, 149,  30],
        [  0,   0,   0,  ..., 272,  97, 273],
        [  0,   0,   0,  ..., 132,  22, 133],
        ...,
        [  0,   0,   0,  ...,   0,   0,   4],
        [  0,   0,   0,  ...,   4, 263, 264],
        [  0,   0,   0,  ..., 179,   8,   2]]) tensor([  4, 274,  17,  50, 158, 212,  11, 105, 159,  22,  33,  76, 111,  15,
        119, 252, 177,  78,  86,  95,   5, 219, 260,  27,  89, 132, 152,  85,
         22,   5, 252, 261])
tensor([[  0,   0,   0,  ...,  37,  38,  39],
        [  0,   0,   0,  ..., 145, 142,   2],
        [  0,   0,   0,  ..., 141, 144,  22],
        ...,
        [  0,   0,   0,  ...,   0,  22,  23],
        [  0,   0,   0,  ...,   3,  12,  13],
        [  0,   0,   0,  ..., 152,  73,  94]]) tensor([ 34, 143, 155,  45, 164,  35,  90, 152, 168,  74,  33, 109, 283,  78,
         22, 151, 170, 214,  86, 151,   2, 117,  24, 110, 253, 201, 275,  19,
         76,  24,  14, 241])
tensor([[  0,   0,   0,  ...,   5, 221,  30],
    

In [34]:
# LSTM architecture
class LSTMModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,100)
    self.lstm=nn.LSTM(100,150,batch_first=True)
    self.fc=nn.Linear(150,vocab_size)

  def forward(self,x):
    embedded=self.embedding(x)
    intermediate_hidden_states,(final_hidden_state,final_cell_state)=self.lstm(embedded)
    output=self.fc(final_hidden_state.squeeze(0))
    return output

In [35]:
model=LSTMModel(len(vocab))

In [36]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [37]:
model.to(device)

LSTMModel(
  (embedding): Embedding(289, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=289, bias=True)
)

In [38]:
epochs=50
learning_rate=0.001
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [39]:
# training loop


for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")
# for epoch in range(epochs):
#   total_loss=0

#   for batch_x,batch_y in dataloader:
#     batch_x=batch_x.to(device)
#     batch_y=batch_y.to(device)
#     optimizer.zero_grad()
#     output=model(batch_x)
#     loss=criterion(output,batch_y)
#     loss.backward()
#     optimizer.step()
#     total_loss+=loss.item()
#   print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader)}")



Epoch: 1, Loss: 165.6339
Epoch: 2, Loss: 145.4152
Epoch: 3, Loss: 133.2655
Epoch: 4, Loss: 120.2035
Epoch: 5, Loss: 107.5852
Epoch: 6, Loss: 95.8674
Epoch: 7, Loss: 85.5550
Epoch: 8, Loss: 75.9719
Epoch: 9, Loss: 67.3577
Epoch: 10, Loss: 59.5616
Epoch: 11, Loss: 52.6786
Epoch: 12, Loss: 46.0830
Epoch: 13, Loss: 40.7709
Epoch: 14, Loss: 35.5792
Epoch: 15, Loss: 31.2402
Epoch: 16, Loss: 27.4177
Epoch: 17, Loss: 24.3590
Epoch: 18, Loss: 21.7713
Epoch: 19, Loss: 19.3393
Epoch: 20, Loss: 17.3652
Epoch: 21, Loss: 15.5472
Epoch: 22, Loss: 14.0142
Epoch: 23, Loss: 12.8413
Epoch: 24, Loss: 11.6214
Epoch: 25, Loss: 10.7837
Epoch: 26, Loss: 10.0584
Epoch: 27, Loss: 9.5397
Epoch: 28, Loss: 8.7586
Epoch: 29, Loss: 8.4093
Epoch: 30, Loss: 7.7355
Epoch: 31, Loss: 7.5239
Epoch: 32, Loss: 7.2111
Epoch: 33, Loss: 6.7452
Epoch: 34, Loss: 6.5530
Epoch: 35, Loss: 6.3190
Epoch: 36, Loss: 5.9396
Epoch: 37, Loss: 5.9163
Epoch: 38, Loss: 5.5412
Epoch: 39, Loss: 5.5145
Epoch: 40, Loss: 5.3608
Epoch: 41, Loss: 5

In [53]:
# prediction
def prediction(model,vocab,text):
  # text -->numerical indices
  numerical_text=text_to_indices(word_tokenize(text.lower()),vocab)

  # padding
  padded_text=[0]*(61-len(numerical_text)) + numerical_text
  padded_text=torch.tensor(padded_text,dtype=torch.long).unsqueeze(0)
  padded_text = padded_text.to(device)

  # send to model
  output=model(padded_text)

  # predicted index
  value,index=torch.max(output,dim=1)


  # merge with text
  return text+' '+list(vocab.keys())[index]

In [54]:
prediction(model,vocab,'The total duration of the course')

'The total duration of the course is'

In [57]:
num_tokens=10
input_text='The total duration of the course'
for i in range(num_tokens):
  output_text=prediction(model,vocab,input_text)
  print(output_text)
  input_text=output_text


The total duration of the course is
The total duration of the course is 7
The total duration of the course is 7 months
The total duration of the course is 7 months .
The total duration of the course is 7 months . so
The total duration of the course is 7 months . so the
The total duration of the course is 7 months . so the total
The total duration of the course is 7 months . so the total course
The total duration of the course is 7 months . so the total course fee
The total duration of the course is 7 months . so the total course fee becomes
